<a href="https://colab.research.google.com/github/thanhsonnguyen-gif/ai-text-detection/blob/main/notebooks/E3_generalization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E3 · LLM chưa thấy · Chủ: Người 1 · Phục vụ Chương 5 (mục 5.3, RQ2)
Huấn luyện chỉ trên văn bản người + LLM cũ (`config.OLD_GENERATORS`), chọn ngưỡng trên validation (cũng chỉ người + LLM cũ),
đo trên test theo từng LLM → **T5.5**.

In [ ]:
# === BOOTSTRAP: giữ nguyên ở đầu mọi notebook ===
OWNER = "Son&Tai"   # tên bạn
REPO  = "https://github.com/thanhsonnguyen-gif/ai-text-detection.git"

from google.colab import drive
drive.mount("/content/drive")
import os, sys, subprocess
if not os.path.exists("/content/repo"):
    url = REPO
    try:   # repo private: lưu Personal Access Token vào Colab Secrets với tên GH_TOKEN
        from google.colab import userdata
        url = REPO.replace("https://", f"https://{userdata.get('GH_TOKEN')}@")
    except Exception:
        pass
    subprocess.run(["git", "clone", "-q", url, "/content/repo"], check=True)
else:   # --ff-only + check=True: pull hỏng thì DỪNG, không âm thầm chạy tiếp với code cũ
    subprocess.run(["git", "-C", "/content/repo", "pull", "-q", "--ff-only"], check=True)
print("Code đang dùng:", subprocess.run(["git", "-C", "/content/repo", "log", "-1", "--format=%h %s (%cd)", "--date=short"],
                                        capture_output=True, text=True).stdout.strip())
for k in [k for k in sys.modules if k == "aitext" or k.startswith("aitext.")]:
    del sys.modules[k]   # chạy lại BOOTSTRAP ⇒ nạp lại aitext mới nhất, không dùng bản cũ còn trong bộ nhớ
if "/content/repo/src" not in sys.path:
    sys.path.insert(0, "/content/repo/src")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "/content/repo/requirements.txt"])

from aitext import config, spark_utils
spark_utils.ensure_pyspark()
spark = spark_utils.get_spark()
config.ensure_dirs()
import time, pandas as pd, matplotlib.pyplot as plt
from pyspark.sql import functions as F
print("Drive:", config.DRIVE_ROOT)

Mounted at /content/drive
Java 21 | PySpark 4.0.5
Drive: /content/drive/MyDrive/BigData_IE212_R11_VB2/AI-Text-Project


In [ ]:
from aitext import data, models, evaluate
train, val, test = data.load_splits(spark)
old = (F.col("label") == 0) | F.col("model").isin(config.OLD_GENERATORS)
tr = data.with_class_weight(train.filter(old)).cache()
m = models.fit(tr, defended=True)                         # dùng regParam đã chốt ở E1
thr = evaluate.thr_at_fpr(evaluate.score(m, val.filter(old), models.text_col(True)))
pred = evaluate.score(m, test, models.text_col(True)).cache()
human = pred.filter("label = 0"); rows = []
for g_ in config.OLD_GENERATORS + config.NEW_GENERATORS:
    sub = human.unionByName(pred.filter(F.col("model") == g_))
    rows.append({"LLM": g_, "nhóm": "đã thấy" if g_ in config.OLD_GENERATORS else "CHƯA thấy",
                 "AUROC": evaluate.auroc(sub), "TPR": evaluate.rate_above(sub, 1, thr)})
t55 = pd.DataFrame(rows); display(t55.round(3))
evaluate.save_table(t55, "T5.5_unseen_generators")
display(t55.groupby("nhóm")[["AUROC", "TPR"]].mean().round(3))

,LLM,nhóm,AUROC,TPR
0,gpt2,đã thấy,0.843,0.606
1,gpt3,đã thấy,0.921,0.722
2,mpt,đã thấy,0.832,0.595
3,mpt-chat,đã thấy,0.979,0.917
4,cohere,đã thấy,0.813,0.415
5,cohere-chat,đã thấy,0.920,0.681
6,gpt4,CHƯA thấy,0.919,0.716
7,chatgpt,CHƯA thấy,0.973,0.897
8,llama-chat,CHƯA thấy,0.960,0.843
9,mistral,CHƯA thấy,0.818,0.480


,AUROC,TPR
nhóm,,
CHƯA thấy,0.928,0.765
đã thấy,0.885,0.656


# T5.5b: tách nhiễu "loại mô hình" khỏi "đã thấy / chưa thấy" (AUROC gộp, không phải trung bình cộng)

In [ ]:
# T5.5b: tách nhiễu "loại mô hình" khỏi "đã thấy / chưa thấy" (AUROC gộp, không phải trung bình cộng)
CHAT = ["gpt3", "mpt-chat", "cohere-chat", "gpt4", "chatgpt", "llama-chat", "mistral-chat"]
human = pred.filter("label = 0"); rows = []
for kind, in_kind in [("chat/instruct", F.col("model").isin(CHAT)), ("base", ~F.col("model").isin(CHAT))]:
    for grp, in_grp in [("đã thấy", F.col("model").isin(config.OLD_GENERATORS)),
                        ("CHƯA thấy", F.col("model").isin(config.NEW_GENERATORS))]:
        ai = pred.filter((F.col("label") == 1) & in_kind & in_grp)
        sub = human.unionByName(ai)
        rows.append({"loại": kind, "nhóm": grp, "số LLM": ai.select("model").distinct().count(),
                     "n AI": ai.count(), "AUROC": evaluate.auroc(sub), "TPR": evaluate.rate_above(sub, 1, thr)})
t55b = pd.DataFrame(rows); display(t55b.round(3))
evaluate.save_table(t55b, "T5.5b_seen_vs_model_type",
                    notes="gpt3 (text-davinci) xếp vào chat/instruct vì đã được tinh chỉnh theo chỉ dẫn.")

,loại,nhóm,số LLM,n AI,AUROC,TPR
0,chat/instruct,đã thấy,3,3070,0.950,0.809
1,chat/instruct,CHƯA thấy,4,4609,0.958,0.846
2,base,đã thấy,3,3945,0.833,0.563
3,base,CHƯA thấy,1,1616,0.818,0.480


## Ghi chú cho Chương 5
- ...